In [89]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import glob
def compute_diversity(mols, verbose=False):
    if verbose:
        print("Computing diversity...")

    if len(mols) == 0:
        return 0

    sims = []
    fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    for i in range(len(fps)):
        sims += DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    return 1 - np.mean(sims)

def compute_diversity2(mols, verbose=False):
    if verbose:
        print("Computing diversity...")

    if len(mols) == 0:
        return 0

    sims = []
    fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    sims = []
    for i in range(len(fps)):
        fps_copy = fps.copy()
        f = fps[i]
        del fps_copy[i]
        sim = DataStructs.BulkTanimotoSimilarity(f, fps_copy)
        sims.append(np.max(sim))
    return sims

def compute_novelty(mols, ref_mols):
    fps_mol = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in mols]
    fps_ref = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(x), 3, 2048) for x in ref_mols]
    sims = []
    for f in fps_mol:
        sim = DataStructs.BulkTanimotoSimilarity(f, fps_ref)
        sims.append(np.max(sim))
    return sims

def harmonic_mean(values):
    n = len(values)
    if any(v == 0 for v in values):
        return 0
    return n / sum(1/v for v in values)
def get_results(dir_path):
    results = []
    topk_df_dict = {}
    all_df_dict = {}
    for i in range(1000,30001,1000):
        path = f'{dir_path}/sampled_mols_{i}.tsv'
        try:
            df = pd.read_table(path)
            all_df_dict[i] = df
        except:
            break
        u_df = df.drop_duplicates('smiles')
        # print(f'u_df: {u_df.shape}')
        all_score_mean = u_df['reward'].mean()
        all_div = compute_diversity(u_df['smiles'])
        all_unique = df.drop_duplicates().shape[0]/df.shape[0]
        high_unique = df[df['reward']>=0.5].drop_duplicates().shape[0]
        all_hm = harmonic_mean([all_score_mean, all_div, all_unique])
        # print(f'all_score_mean: {all_score_mean}, all_div: {all_div}, all_unique: {all_unique}, all_harmonic_mean: {all_hm}')
        topk_df = u_df.nlargest(100, 'reward')
        topk_df_dict[i] = topk_df
        # print(topk_df.head(1))
        topk_score_mean = topk_df['reward'].mean()
        topk_score_median = topk_df['reward'].median()
        topk_div = compute_diversity(topk_df['smiles'])
        topk_unique = topk_df.drop_duplicates().shape[0]/topk_df.shape[0]
        topk_success = topk_df[topk_df['reward']>=0.5].shape[0]/topk_df.shape[0]
        topk_tlen = topk_df['traj_len'].mean()
        top_hm = harmonic_mean([topk_score_mean, topk_div, all_unique])
        # print(f'topk_score_mean: {topk_score_mean}, topk_div: {topk_div}, topk_unique: {topk_unique}, topk_harmonic_mean: {top_hm}')
        results.append([i, topk_tlen, high_unique,all_score_mean, all_div, all_unique, topk_score_mean, topk_div, topk_success, top_hm])
    results_df = pd.DataFrame(results, columns=['i', 'topk_tlen', 'high_unique', 'all_score_mean', 'all_div', 'all_unique', 'topk_score_mean', 'topk_div', 'topk_success','top_hm'])
    return results_df, topk_df_dict, all_df_dict

def get_results_topk(dir_path):
    results = []
    topk_df_dict = {}
    all_df_dict = {}
    for i in range(1000,30001,1000):
        path = f'{dir_path}/sampled_mols_{i}.tsv'
        try:
            df = pd.read_table(path)
            all_df_dict[i] = df
        except:
            break
        u_df = df.drop_duplicates('smiles')
        all_unique = df.drop_duplicates().shape[0]/df.shape[0]
        high_unique = df[df['reward']>=0.5].drop_duplicates().shape[0]
        topk_df = u_df.nlargest(100, 'reward')
        topk_df_dict[i] = topk_df
        # print(topk_df.head(1))
        topk_score_mean = topk_df['reward'].mean()
        topk_score_median = topk_df['reward'].median()
        topk_div = compute_diversity(topk_df['smiles'])
        topk_unique = topk_df.drop_duplicates().shape[0]/topk_df.shape[0]
        topk_success = topk_df[topk_df['reward']>=0.5].shape[0]/topk_df.shape[0]
        top_hm = harmonic_mean([topk_score_mean, topk_div, all_unique, topk_success])
        # print(f'topk_score_mean: {topk_score_mean}, topk_div: {topk_div}, topk_unique: {topk_unique}, topk_harmonic_mean: {top_hm}')
        results.append([i, high_unique, all_unique, topk_score_mean, topk_div, topk_success, top_hm])
    results_df = pd.DataFrame(results, columns=['i', 'high_unique', 'all_unique', 'topk_score_mean', 'topk_div', 'topk_success', 'top_hm'])
    return results_df, topk_df_dict, all_df_dict

In [91]:
oracle='jnk3'

method='DB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique', 'topk_tlen']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success', 'topk_tlen']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success', 'topk_tlen']]
# display(df)
print(oracle)
print(method)
display(df.T.describe())

method='TB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='SubTB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='FM'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='LeakGFN'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

jnk3
DB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.000000,3.000000
mean,14666.666667,0.623326,0.397067,0.772649,1.0,0.033333,3.333333
std,5773.502692,0.002750,0.001710,0.011853,0.0,0.005774,0.577350
min,8000.000000,0.621141,0.395100,0.759518,1.0,0.030000,3.000000
25%,13000.000000,0.621782,0.396500,0.767694,1.0,0.030000,3.000000
50%,18000.000000,0.622422,0.397900,0.775869,1.0,0.030000,3.000000
75%,18000.000000,0.624418,0.398050,0.779214,1.0,0.035000,3.500000
max,18000.000000,0.626415,0.398200,0.782559,1.0,0.040000,4.000000


TB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.000000,3.000000
mean,14333.333333,0.623938,0.398400,0.770407,1.0,0.023333,2.333333
std,10115.993937,0.003014,0.004078,0.002909,0.0,0.005774,0.577350
min,8000.000000,0.621871,0.395800,0.768705,1.0,0.020000,2.000000
25%,8500.000000,0.622209,0.396050,0.768727,1.0,0.020000,2.000000
50%,9000.000000,0.622547,0.396300,0.768750,1.0,0.020000,2.000000
75%,17500.000000,0.624972,0.399700,0.771257,1.0,0.025000,2.500000
max,26000.000000,0.627397,0.403100,0.773765,1.0,0.030000,3.000000


SubTB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,8333.333333,0.619879,0.394567,0.767019,0.999667,0.030000,3.000000
std,577.350269,0.008953,0.012474,0.008665,0.000577,0.026458,2.645751
min,8000.000000,0.609604,0.380600,0.757182,0.999000,0.010000,1.000000
25%,8000.000000,0.616814,0.389550,0.763770,0.999500,0.015000,1.500000
50%,8000.000000,0.624024,0.398500,0.770358,1.000000,0.020000,2.000000
75%,8500.000000,0.625016,0.401550,0.771938,1.000000,0.040000,4.000000
max,9000.000000,0.626009,0.404600,0.773519,1.000000,0.060000,6.000000


FM


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,10666.666667,0.402934,0.399433,0.642827,0.444333,0.230000,23.000000
std,15885.003410,0.218182,0.139922,0.116175,0.467685,0.281603,28.160256
min,1000.000000,0.195059,0.241300,0.518847,0.101000,0.020000,2.000000
25%,1500.000000,0.289332,0.345550,0.589649,0.178000,0.070000,7.000000
50%,2000.000000,0.383606,0.449800,0.660451,0.255000,0.120000,12.000000
75%,15500.000000,0.506871,0.478500,0.704817,0.616000,0.335000,33.500000
max,29000.000000,0.630137,0.507200,0.749183,0.977000,0.550000,55.000000


LeakGFN


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.0,3.000000
mean,22333.333333,0.652885,0.599033,0.540870,0.930333,1.0,322.666667
std,7234.178138,0.067735,0.071026,0.061163,0.062883,0.0,198.015993
min,14000.000000,0.574694,0.518000,0.470774,0.858000,1.0,109.000000
25%,20000.000000,0.632528,0.573300,0.519612,0.909500,1.0,234.000000
50%,26000.000000,0.690361,0.628600,0.568450,0.961000,1.0,359.000000
75%,26500.000000,0.691981,0.639550,0.575918,0.966500,1.0,429.500000
max,27000.000000,0.693600,0.650500,0.583386,0.972000,1.0,500.000000


In [88]:
oracle='gsk3b'

method='DB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(df)
print(oracle)
print(method)
display(df.T.describe())

method='TB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='SubTB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='FM'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='LeakGFN'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

gsk3b
DB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.000000,3.000000
mean,14333.333333,0.673345,0.438298,0.852128,1.0,0.133333,13.333333
std,14640.127504,0.001278,0.004497,0.011005,0.0,0.032146,3.214550
min,1000.000000,0.671893,0.433294,0.842734,1.0,0.110000,11.000000
25%,6500.000000,0.672868,0.436447,0.846075,1.0,0.115000,11.500000
50%,12000.000000,0.673843,0.439600,0.849416,1.0,0.120000,12.000000
75%,21000.000000,0.674072,0.440800,0.856826,1.0,0.145000,14.500000
max,30000.000000,0.674300,0.442000,0.864236,1.0,0.170000,17.000000


TB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.00,3.0
mean,19666.666667,0.675024,0.440367,0.852719,1.0,0.12,12.0
std,4509.249753,0.008776,0.012522,0.004548,0.0,0.04,4.0
min,15000.000000,0.669307,0.432400,0.847481,1.0,0.08,8.0
25%,17500.000000,0.669972,0.433150,0.851246,1.0,0.10,10.0
50%,20000.000000,0.670637,0.433900,0.855011,1.0,0.12,12.0
75%,22000.000000,0.677883,0.444350,0.855338,1.0,0.14,14.0
max,24000.000000,0.685129,0.454800,0.855666,1.0,0.16,16.0


SubTB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.00000,3.000000
mean,13666.666667,0.685981,0.457033,0.845544,0.999667,0.18000,18.000000
std,11590.225767,0.014728,0.022426,0.010918,0.000577,0.10583,10.583005
min,3000.000000,0.671498,0.436400,0.832977,0.999000,0.10000,10.000000
25%,7500.000000,0.678499,0.445100,0.841968,0.999500,0.12000,12.000000
50%,12000.000000,0.685500,0.453800,0.850959,1.000000,0.14000,14.000000
75%,19000.000000,0.693222,0.467350,0.851827,1.000000,0.22000,22.000000
max,26000.000000,0.700943,0.480900,0.852695,1.000000,0.30000,30.000000


FM


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,3666.666667,0.716075,0.593733,0.807091,0.814333,0.926667,151.333333
std,2081.665999,0.070873,0.028676,0.020781,0.206379,0.127017,66.890458
min,2000.000000,0.634481,0.563400,0.783162,0.578000,0.780000,78.000000
25%,2500.000000,0.692946,0.580400,0.800335,0.742000,0.890000,122.500000
50%,3000.000000,0.751411,0.597400,0.817508,0.906000,1.000000,167.000000
75%,4500.000000,0.756872,0.608900,0.819056,0.932500,1.000000,188.000000
max,6000.000000,0.762333,0.620400,0.820604,0.959000,1.000000,209.000000


LeakGFN


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.00000,3.000000,3.000000,3.0,3.000000
mean,19000.000000,0.760266,0.62600,0.804196,0.910333,1.0,204.666667
std,14177.446879,0.008359,0.04267,0.014112,0.044061,0.0,89.890674
min,3000.000000,0.753590,0.58410,0.788040,0.881000,1.0,120.000000
25%,13500.000000,0.755578,0.60430,0.799235,0.885000,1.0,157.500000
50%,24000.000000,0.757565,0.62450,0.810430,0.889000,1.0,195.000000
75%,27000.000000,0.763603,0.64695,0.812273,0.925000,1.0,247.000000
max,30000.000000,0.769641,0.66940,0.814117,0.961000,1.0,299.000000


In [87]:
oracle='drd2'

method='DB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(df)
print(oracle)
print(method)
display(df.T.describe())

method='TB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='SubTB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='FM'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='LeakGFN'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

drd2
DB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,26000.000000,0.934314,0.944239,0.868214,1.0,1.0,380.000000
std,3464.101615,0.002287,0.009668,0.002343,0.0,0.0,64.280635
min,24000.000000,0.931862,0.934328,0.865661,1.0,1.0,306.000000
25%,24000.000000,0.933277,0.939536,0.867188,1.0,1.0,359.000000
50%,24000.000000,0.934692,0.944744,0.868714,1.0,1.0,412.000000
75%,27000.000000,0.935541,0.949194,0.869490,1.0,1.0,417.000000
max,30000.000000,0.936389,0.953644,0.870266,1.0,1.0,422.000000


TB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,26666.666667,0.933059,0.942900,0.866052,1.0,1.0,401.666667
std,2886.751346,0.000329,0.001677,0.002264,0.0,0.0,29.737743
min,25000.000000,0.932784,0.941111,0.864046,1.0,1.0,384.000000
25%,25000.000000,0.932877,0.942132,0.864825,1.0,1.0,384.500000
50%,25000.000000,0.932971,0.943154,0.865605,1.0,1.0,385.000000
75%,27500.000000,0.933197,0.943795,0.867056,1.0,1.0,410.500000
max,30000.000000,0.933423,0.944436,0.868507,1.0,1.0,436.000000


SubTB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.0,3.000000,3.000000
mean,12000.0,0.845159,0.716922,0.872484,1.0,0.983333,129.666667
std,2000.0,0.039106,0.088638,0.003202,0.0,0.028868,49.217206
min,10000.0,0.814630,0.648923,0.869611,1.0,0.950000,95.000000
25%,11000.0,0.823120,0.666799,0.870758,1.0,0.975000,101.500000
50%,12000.0,0.831609,0.684674,0.871905,1.0,1.000000,108.000000
75%,13000.0,0.860424,0.750922,0.873920,1.0,1.000000,147.000000
max,14000.0,0.889238,0.817170,0.875935,1.0,1.000000,186.000000


FM


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,2666.666667,0.775111,0.661045,0.800229,0.939333,0.740000,91.666667
std,2081.665999,0.041215,0.145937,0.052331,0.027062,0.238956,53.715299
min,1000.000000,0.740806,0.546322,0.744573,0.909000,0.530000,53.000000
25%,1500.000000,0.752251,0.578916,0.776126,0.928500,0.610000,61.000000
50%,2000.000000,0.763696,0.611511,0.807679,0.948000,0.690000,69.000000
75%,3500.000000,0.792263,0.718406,0.828058,0.954500,0.845000,111.000000
max,5000.000000,0.820829,0.825302,0.848437,0.961000,1.000000,153.000000


LeakGFN


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,2000.0,0.796960,0.675142,0.826218,0.962333,0.823333,117.666667
std,0.0,0.070043,0.150128,0.036737,0.021385,0.305996,79.027421
min,2000.0,0.719217,0.522845,0.785091,0.939000,0.470000,47.000000
25%,2000.0,0.767867,0.601212,0.811438,0.953000,0.735000,75.000000
50%,2000.0,0.816516,0.679580,0.837785,0.967000,1.000000,103.000000
75%,2000.0,0.835831,0.751291,0.846782,0.974000,1.000000,153.000000
max,2000.0,0.855146,0.823001,0.855780,0.981000,1.000000,203.000000


In [86]:
oracle='qed'

method='DB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(df)
print(oracle)
print(method)
display(df.T.describe())

method='TB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='SubTB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='FM'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='LeakGFN'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

qed
DB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,4333.333333,0.897397,0.819294,0.897821,1.0,1.0,542.333333
std,3214.550254,0.043862,0.103315,0.004359,0.0,0.0,145.850380
min,2000.000000,0.846757,0.700206,0.893906,1.0,1.0,391.000000
25%,2500.000000,0.884371,0.786464,0.895473,1.0,1.0,472.500000
50%,3000.000000,0.921985,0.872723,0.897040,1.0,1.0,554.000000
75%,5500.000000,0.922717,0.878838,0.899779,1.0,1.0,618.000000
max,8000.000000,0.923449,0.884953,0.902518,1.0,1.0,682.000000


TB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.0,3.000000
mean,4333.333333,0.899194,0.835911,0.886580,0.999667,1.0,695.333333
std,4932.882862,0.047342,0.119648,0.007266,0.000577,0.0,248.339150
min,1000.000000,0.844555,0.697755,0.879138,0.999000,1.0,409.000000
25%,1500.000000,0.884782,0.801269,0.883042,0.999500,1.0,617.000000
50%,2000.000000,0.925009,0.904784,0.886947,1.000000,1.0,825.000000
75%,6000.000000,0.926513,0.904990,0.890302,1.000000,1.0,838.500000
max,10000.000000,0.928017,0.905196,0.893657,1.000000,1.0,852.000000


SubTB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,6666.666667,0.845148,0.701671,0.889478,1.0,1.0,363.666667
std,4163.331999,0.006934,0.014405,0.003902,0.0,0.0,75.062196
min,2000.000000,0.840320,0.692734,0.885642,1.0,1.0,277.000000
25%,5000.000000,0.841175,0.693362,0.887496,1.0,1.0,341.500000
50%,8000.000000,0.842029,0.693991,0.889350,1.0,1.0,406.000000
75%,9000.000000,0.847561,0.706140,0.891396,1.0,1.0,407.000000
max,10000.000000,0.853093,0.718289,0.893442,1.0,1.0,408.000000


FM


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0000,3.0,3.000000
mean,8333.333333,0.926022,0.890672,0.898584,0.9960,1.0,900.666667
std,2886.751346,0.002600,0.007289,0.002017,0.0010,0.0,15.534907
min,5000.000000,0.923212,0.883742,0.896603,0.9950,1.0,888.000000
25%,7500.000000,0.924863,0.886871,0.897558,0.9955,1.0,892.000000
50%,10000.000000,0.926515,0.890000,0.898513,0.9960,1.0,896.000000
75%,10000.000000,0.927428,0.894138,0.899574,0.9965,1.0,907.000000
max,10000.000000,0.928340,0.898275,0.900635,0.9970,1.0,918.000000


LeakGFN


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,8000.000000,0.926464,0.903759,0.883707,1.0,1.0,830.000000
std,2645.751311,0.001925,0.005100,0.004331,0.0,0.0,34.698703
min,5000.000000,0.924603,0.898142,0.879237,1.0,1.0,800.000000
25%,7000.000000,0.925472,0.901588,0.881617,1.0,1.0,811.000000
50%,9000.000000,0.926340,0.905034,0.883998,1.0,1.0,822.000000
75%,9500.000000,0.927394,0.906567,0.885941,1.0,1.0,845.000000
max,10000.000000,0.928447,0.908100,0.887885,1.0,1.0,868.000000


In [85]:
oracle='sa'

method='DB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
# display(df)
print(oracle)
print(method)
display(df.T.describe())

method='TB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='SubTB'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='FM'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

method='LeakGFN'
df = pd.DataFrame()
path = glob.glob(f'./checkpoints/{method}/seed_1/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_1'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']]
path = glob.glob(f'./checkpoints/{method}/seed_2/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_2'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
path = glob.glob(f'./checkpoints/{method}/seed_3/{oracle}/*')[-1]
results_df, topk_df_dict, all_df_dict = get_results(path)
df['seed_3'] = results_df.sort_values(by='top_hm', ascending=False)[['i', 'top_hm','topk_score_mean','topk_div','all_unique','topk_success','high_unique']].head(1).T
results_df.sort_values(by='top_hm', ascending=False)[['high_unique','top_hm','all_unique','topk_div','topk_score_mean','topk_success']]
print(method)
display(df.T.describe())

sa
DB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.0,3.0,3.000000
mean,1000.0,0.863746,0.725821,0.912931,1.0,1.0,859.333333
std,0.0,0.006972,0.011964,0.005109,0.0,0.0,3.055050
min,1000.0,0.859720,0.717613,0.908099,1.0,1.0,856.000000
25%,1000.0,0.859721,0.718957,0.910256,1.0,1.0,858.000000
50%,1000.0,0.859721,0.720301,0.912414,1.0,1.0,860.000000
75%,1000.0,0.865759,0.729925,0.915346,1.0,1.0,861.000000
max,1000.0,0.871797,0.739549,0.918279,1.0,1.0,862.000000


TB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.000000,3.0,3.000000
mean,1000.0,0.867129,0.734454,0.910990,0.999667,1.0,900.666667
std,0.0,0.002147,0.006926,0.005872,0.000577,0.0,36.460024
min,1000.0,0.864840,0.729785,0.905066,0.999000,1.0,868.000000
25%,1000.0,0.866144,0.730475,0.908080,0.999500,1.0,881.000000
50%,1000.0,0.867448,0.731165,0.911094,1.000000,1.0,894.000000
75%,1000.0,0.868273,0.736788,0.913952,1.000000,1.0,917.000000
max,1000.0,0.869099,0.742412,0.916809,1.000000,1.0,940.000000


SubTB


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.000000,3.000000,3.000000,3.000000,3.000,3.0,3.000000
mean,1333.333333,0.880059,0.764978,0.908855,0.999,1.0,918.000000
std,577.350269,0.008422,0.019142,0.004501,0.000,0.0,37.469988
min,1000.000000,0.872846,0.751321,0.904942,0.999,1.0,876.000000
25%,1000.000000,0.875431,0.754038,0.906395,0.999,1.0,903.000000
50%,1000.000000,0.878016,0.756755,0.907849,0.999,1.0,930.000000
75%,1500.000000,0.883665,0.771806,0.910811,0.999,1.0,939.000000
max,2000.000000,0.889314,0.786856,0.913773,0.999,1.0,948.000000


FM


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.000000,3.0,3.000000
mean,1000.0,0.921750,0.908719,0.890380,0.970000,1.0,969.333333
std,0.0,0.003665,0.004542,0.012418,0.008718,0.0,9.451631
min,1000.0,0.917523,0.903675,0.879957,0.964000,1.0,962.000000
25%,1000.0,0.920607,0.906836,0.883510,0.965000,1.0,964.000000
50%,1000.0,0.923691,0.909998,0.887062,0.966000,1.0,966.000000
75%,1000.0,0.923863,0.911241,0.895591,0.973000,1.0,973.000000
max,1000.0,0.924035,0.912485,0.904119,0.980000,1.0,980.000000


LeakGFN


,i,top_hm,topk_score_mean,topk_div,all_unique,topk_success,high_unique
count,3.0,3.000000,3.000000,3.000000,3.000000,3.0,3.000000
mean,1000.0,0.923001,0.910447,0.887918,0.975333,1.0,974.000000
std,0.0,0.002932,0.010753,0.010470,0.016653,0.0,16.370706
min,1000.0,0.921216,0.898649,0.879442,0.962000,1.0,960.000000
25%,1000.0,0.921310,0.905821,0.882066,0.966000,1.0,965.000000
50%,1000.0,0.921404,0.912994,0.884689,0.970000,1.0,970.000000
75%,1000.0,0.923894,0.916346,0.892155,0.982000,1.0,981.000000
max,1000.0,0.926385,0.919697,0.899621,0.994000,1.0,992.000000
